# LHR workflow demo

This notebook reads the 20260110 DEIMOS tables, keeps only non-variable member stars, lets the user choose which systems to analyze, computes the likelihood-ratio statistic (LHR), and then reports a final summary table with the resolved/unresolved classification and the Gaussian dispersion estimate or upper limit.

## Notes

- The user chooses which systems to fit by updating `selected_systems`.
- The system is considered resolved only when `lhr > threshold`, where `threshold = X - N - N * log(X / N)` and `X` is the 99th percentile of a chi-square variable with `N - 1` degrees of freedom.
- Resolved systems report a posterior mode with a 68% maximum-density interval; unresolved systems report a 90% upper limit on the velocity dispersion.

# Pictor 2

In [56]:
import os
import numpy as np
import pandas as pd
import scipy.stats as st
from scipy.optimize import minimize
import emcee

# Gaussian likelihood and MCMC helpers.

def mlnL(theta, vel, vel_err):
    mu, sigma = theta
    var = sigma**2 + vel_err**2
    return 0.5 * np.sum(np.log(2 * np.pi * var) + (vel - mu)**2 / var)

def MLE(vlist, errlist):
    seed = np.array([np.mean(vlist), np.std(vlist)])
    res = minimize(
        mlnL,
        seed,
        args=(vlist, errlist),
        method='Nelder-Mead',
        options={'maxiter': 5000},
    )
    return float(res.x[0]), float(abs(res.x[1]))

def MLE0(vlist, errlist):
    seed = np.mean(vlist)
    res = minimize(
        lambda x: mlnL([x, 0.0], vlist, errlist),
        seed,
        method='Nelder-Mead',
        options={'maxiter': 5000},
    )
    return float(res.x[0])

def lhr_threshold(n, chi2_percentile=0.99):
    """Threshold for the LHR test.
    X - N - N * log(X / N), where X is the chi-square quantile with
    df = N - 1.
    """
    n = int(n)
    if n < 2:
        return np.nan
    X = st.chi2.ppf(chi2_percentile, df=n - 1)
    return float(X - n - n * np.log(X / n))

def peak_interval(samples, alpha=0.32):
    """Posterior mode and a 68% maximum-density interval."""
    x = np.sort(np.asarray(samples, dtype=float))
    if len(x) == 0:
        return np.nan, [np.nan, np.nan], 0.0
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    if mad == 0:
        mad = 1e-6
    mask = (x > med - 5.0 * mad) & (x < med + 5.0 * mad)
    x2 = x[mask]
    if len(x2) < 2:
        mode = float(np.median(x))
        return mode, [mode, mode], 0.0
    kde = st.gaussian_kde(x2)
    grid = np.linspace(np.min(x2), np.max(x2), 1000)
    pdf = kde.evaluate(grid)
    mode = float(grid[np.argmax(pdf)])
    n = len(x)
    window = int(np.rint((1.0 - alpha) * n))
    if window < 1:
        return mode, [x[0], x[-1]], float(np.max(pdf))
    starts = x[:n - window]
    ends = x[window:]
    widths = ends - starts
    in_window = (mode >= starts) & (mode <= ends)
    if np.any(in_window):
        starts = starts[in_window]
        ends = ends[in_window]
        widths = ends - starts
        idx = np.argmin(widths)
        return mode, [float(starts[idx]), float(ends[idx])], float(np.max(pdf))
    lo = float(np.quantile(x, 0.16))
    hi = float(np.quantile(x, 0.84))
    return mode, [lo, hi], float(np.max(pdf))

def lnprior(theta, vel, lower=1e-3, upper=20.0):
    mu, sigma = theta
    if sigma < lower or sigma > upper:
        return -np.inf
    if not (vel.min() < mu < vel.max()):
        return -np.inf
    return 0.0

def lnlike(theta, vel, vel_err):
    mu, sigma = theta
    var = sigma**2 + vel_err**2
    return -0.5 * np.sum(np.log(2 * np.pi * var) + (vel - mu)**2 / var)

def lnprob(theta, vel, vel_err, lower=1e-3, upper=20.0):
    lp = lnprior(theta, vel, lower=lower, upper=upper)
    if not np.isfinite(lp):
        return -np.inf
    return lp + lnlike(theta, vel, vel_err)

def mcmc(vel, vel_err, lower=1e-3, upper=20.0, nwalkers=20, nburn=200, nsteps=800):
    vel = np.asarray(vel, dtype=float)
    vel_err = np.asarray(vel_err, dtype=float)
    good = np.isfinite(vel) & np.isfinite(vel_err)
    vel = vel[good]
    vel_err = vel_err[good]
    if len(vel) < 3:
        return None
    mean, std = np.mean(vel), np.std(vel)
    mu0 = np.random.normal(loc=mean, scale=max(std, 1.0), size=nwalkers)
    sigma0 = np.abs(np.random.normal(loc=max(std, 0.5), scale=1.0, size=nwalkers))
    sigma0 = np.clip(sigma0, lower, upper)
    start = np.column_stack([mu0, sigma0])
    sampler = emcee.EnsembleSampler(
        nwalkers,
        2,
        lnprob,
        args=(vel, vel_err, lower, upper),
        threads=1,
    )
    sampler.run_mcmc(start, nburn, progress=False)
    sampler.reset()
    sampler.run_mcmc(start, nsteps, progress=False)
    chain = sampler.get_chain(discard=0, thin=1, flat=True)
    samples = pd.DataFrame(chain, columns=['mu', 'sigma'])
    return samples[(samples['sigma'] >= lower) & (samples['sigma'] <= upper)].copy()

def summarize_one_system(gal, vel, vel_err):
    vel = np.asarray(vel, dtype=float)
    vel_err = np.asarray(vel_err, dtype=float)
    if len(vel) < 2:
        return {
            'UFD': gal,
            'N': len(vel),
            'threshold': np.nan,
            'lhr': np.nan,
            'decision': 'unresolved',
            'dispersion': 'n/a',
        }

    mu_hat, sigma_hat = MLE(vel, vel_err)
    mu0 = MLE0(vel, vel_err)
    like_v = mlnL([mu_hat, sigma_hat], vel, vel_err)
    like_0 = mlnL([mu0, 0.0], vel, vel_err)
    lhr = 2.0 * (like_0 - like_v)
    threshold = lhr_threshold(len(vel), chi2_percentile=0.99)
    resolved = bool(lhr > threshold)

    samples = mcmc(vel, vel_err, lower=1e-3, upper=20.0, nwalkers=20, nburn=100, nsteps=500)
    if samples is None or len(samples) == 0:
        disp_text = 'n/a'
    else:
        sigma_post = samples['sigma'].to_numpy()
        if resolved:
            peak, interval, _ = peak_interval(sigma_post, alpha=0.32)
            lo, hi = interval
            disp_text = f'{peak:.2f}_'+"{"+f"{peak-lo:.2f}"+"}^{"+f'{hi-peak:.2f}'+"}"
        else:
            upper_limit = float(np.percentile(sigma_post, 90))
            disp_text = f'<{upper_limit:.2f}'

    return {
        #'galaxy': "\text{" + gal + "}",
        'galaxy': gal,
        'N': int(len(vel)),
        'threshold': "{:.2f}".format(threshold),
        'lhr': "{:.2f}".format(lhr),
        #'decision': '\text{Resolved}' if resolved else '\text{Unresolved}',
        'decision': 'Resolved' if resolved else 'Unresolved',
        'dispersion': disp_text,
    }

In [82]:
# read in pictor2 data
from astropy.io import fits

with fits.open('/Users/pierrethibodeaux/Downloads/data_to_share/pictor_2_imacs_combined_catalog.fits') as hdul:
    pictor2_data = hdul[1].data
    member_mask = pictor2_data['member'] == 1
    member_data = pictor2_data[member_mask]
    #remove binary 10286300001343
    binary_mask = member_data['objname'] != 10286300001343
    member_data = member_data[binary_mask]

    #print(member_data['objname'],member_data['vlos'],member_data['vlos_error'])

rows = []
rows.append(summarize_one_system("Pic2", member_data['vlos'], member_data['vlos_error']))

summary_0 = pd.DataFrame(rows, columns=['galaxy', 'N', 'threshold', 'lhr', 'decision', 'dispersion'])
summary_0['instrument']='Magellan/IMACS'
summary_0['source']='\citet{pace_spectroscopic_2025}'

for line in summary_0[['galaxy', 'N', 'threshold', 'lhr', 'decision', 'dispersion']].to_string(index=False).splitlines()[1:]:
    print(" & ".join(line.split()),"\\\\")

/Users/pierrethibodeaux/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]


Pic2 & 12 & 4.05 & 27.97 & Resolved & 3.62_{1.00}^{1.24} \\


# DEIMOS systems from Geha 2026

In [ ]:


# Read the 20260110 CSV files from the current working directory.
table_files = [
    #'table3A_20260110.csv',
    'table5A_20260110.csv',
]

tables = []
for fn in table_files:
    if os.path.exists(fn):
        tables.append(pd.read_csv(fn))

if not tables:
    raise FileNotFoundError('No 20260110 CSV files were found in this directory.')

df = pd.concat(tables, ignore_index=True, sort=False)
print(f'Loaded {len(df)} rows from the 20260110 tables.')

Loaded 24436 rows from the 20260110 tables.


In [3]:
# Keep only non-variable member stars.
# The user can adjust this filter before performing the LHR analysis.

if 'Pmem_novar' in df.columns:
    df_sel = df[df['Pmem_novar'] > 0.5].copy()
else:
    df_sel = df[df.get('Pmem', 0.0) > 0.5].copy()

if 'Var' in df_sel.columns:
    df_sel = df_sel[df_sel['Var'] != 1].copy()

if 'system_name' not in df_sel.columns:
    raise KeyError('The CSV files do not contain a system_name column.')

df_sel['system_name'] = df_sel['system_name'].astype(str)
print(f"After filtering, {len(df_sel)} stars remain across {df_sel['system_name'].nunique()} systems.")

available_systems = sorted(df_sel['system_name'].unique())
print('Available systems:')
for gal in available_systems[:20]:
    print(' -', gal)

After filtering, 11014 stars remain across 78 systems.
Available systems:
 - Aqr2
 - Aqr3
 - Boo1
 - Boo2
 - Boo3
 - CB
 - CVn1
 - CVn2
 - Cet3
 - Col1
 - Dra
 - Dra2
 - Eri
 - Eri4
 - For
 - Herc
 - Hyd2
 - K1
 - K2
 - Lae1


In [22]:
# Choose the systems to include in the analysis.
# Edit this list to narrow the sample to the galaxies you want to study.
selected_systems = sorted(df_sel['system_name'].unique())
# Example custom subset:
# selected_systems = ['Boo1', 'Dra2', 'Seg1', 'Wil1']
selected_systems = ["Aqr2","Aqr3","Boo1","Boo2","Boo3","CVn1","CVn2","Col1","CB","Dra","Dra2","Eri4","Herc","Hyd2","Leo1","Leo2","Leo4","Leo5","Leo6","Peg3","Peg4","Pisc2","Seg1","Seg2","Tri2","UMa1","UMa2","UMi","W1"]

for selected in selected_systems:
    if selected not in available_systems:
        raise ValueError(f'Selected system "{selected}" is not in the available systems.')
print('selected_systems =', selected_systems)

selected_systems = ['Aqr2', 'Aqr3', 'Boo1', 'Boo2', 'Boo3', 'CVn1', 'CVn2', 'Col1', 'CB', 'Dra', 'Dra2', 'Eri4', 'Herc', 'Hyd2', 'Leo1', 'Leo2', 'Leo4', 'Leo5', 'Leo6', 'Peg3', 'Peg4', 'Pisc2', 'Seg1', 'Seg2', 'Tri2', 'UMa1', 'UMa2', 'UMi', 'W1']


In [ ]:

per_system = {}
for gal, gdf in df_sel.groupby('system_name'):
    vel = np.asarray(gdf['v'], dtype=float)
    verr = np.asarray(gdf['v_err'], dtype=float)
    per_system[gal] = {'v': vel, 'verr': verr, 'N': int(len(vel))}

rows = []
for gal in selected_systems:
    if gal not in per_system:
        continue
    rows.append(summarize_one_system(gal, per_system[gal]['v'], per_system[gal]['verr']))

summary = pd.DataFrame(rows, columns=['galaxy', 'N', 'threshold', 'lhr', 'decision', 'dispersion'])

/Users/pierrethibodeaux/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/Users/pierrethibodeaux/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/Users/pierrethibodeaux/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/Users/pierrethibodeaux/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/Users/pierrethibodeaux/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/Users/pierrethibodeaux/anaconda3/lib/py

In [ ]:
summary = summary.sort_values('galaxy', ascending=True).reset_index(drop=True)
summary['instrument']='Keck/DEIMOS'
summary['source']="\citet{geha_keckdeimos_2026-1}"
#print(summary[['galaxy', 'N', 'threshold', 'lhr', 'resolved', 'dispersion_summary']].to_string(index=False))
for line in summary[['galaxy', 'N', 'threshold', 'lhr', 'decision', 'dispersion']].to_string(index=False).splitlines()[1:]:
    print(" & ".join(line.split()),"\\\\")

Aqr2 & 8 & 3.78 & 26.69 & Resolved & 4.29_{1.20}^{2.50} \\
Aqr3 & 11 & 4.00 & 0.00 & Unresolved & <3.22 \\
Boo1 & 92 & 4.88 & 118.57 & Resolved & 3.30_{0.49}^{0.49} \\
Boo2 & 17 & 4.25 & 7.65 & Resolved & 1.69_{0.49}^{0.92} \\
Boo3 & 16 & 4.21 & 7.52 & Resolved & 4.96_{1.72}^{2.06} \\
CB & 82 & 4.85 & 111.84 & Resolved & 3.40_{0.50}^{0.61} \\
CVn1 & 254 & 5.08 & 3319.30 & Resolved & 7.56_{0.37}^{0.54} \\
CVn2 & 30 & 4.51 & 180.34 & Resolved & 5.57_{1.13}^{0.95} \\
Col1 & 10 & 3.93 & 25.71 & Resolved & 5.12_{1.39}^{2.41} \\
Dra & 999 & 5.24 & 24628.56 & Resolved & 9.56_{0.22}^{0.31} \\
Dra2 & 25 & 4.43 & 0.00 & Unresolved & <2.65 \\
Eri4 & 19 & 4.30 & 69.62 & Resolved & 4.06_{0.65}^{1.26} \\
Herc & 43 & 4.65 & 13.07 & Resolved & 2.21_{0.68}^{0.61} \\
Hyd2 & 12 & 4.05 & 0.00 & Unresolved & <4.15 \\
Leo1 & 795 & 5.22 & 19749.61 & Resolved & 9.23_{0.25}^{0.27} \\
Leo2 & 305 & 5.11 & 4801.95 & Resolved & 7.27_{0.26}^{0.45} \\
Leo4 & 24 & 4.42 & 37.59 & Resolved & 3.26_{0.87}^{0.63} \\
Leo5 

# DEIMOS Summary Table
%%latex

$$
\begin{array}{c|cccrc}
\text{UFD} & N & \text{Threshold} & \text{LHR} & \text{Decision} & \text{Dispersion} \\ \hline
\text{Aqr2} & 8 & 3.78 & 26.69 & \text{Resolved} & 4.48_{1.56}^{2.33} \\
\text{Aqr3} & 11 & 4.00 & 0.00 & \text{Unresolved} & <3.06 \\
\text{Boo1} & 92 & 4.88 & 118.57 & \text{Resolved} & 3.21_{0.39}^{0.51} \\
\text{Boo2} & 17 & 4.25 & 7.65 & \text{Resolved} & 1.71_{0.51}^{0.99} \\
\text{Boo3} & 16 & 4.21 & 7.52 & \text{Resolved} & 4.94_{1.41}^{2.46} \\
\text{CB} & 82 & 4.85 & 111.84 & \text{Resolved} & 3.27_{0.54}^{0.57} \\
\text{CVn1} & 254 & 5.08 & 3319.30 & \text{Resolved} & 7.68_{0.50}^{0.40} \\
\text{CVn2} & 30 & 4.51 & 180.34 & \text{Resolved} & 5.30_{0.79}^{1.32} \\
\text{Col1} & 10 & 3.93 & 25.71 & \text{Resolved} & 5.42_{1.84}^{1.92} \\
\text{Dra2} & 25 & 4.43 & 0.00 & \text{Unresolved} & <2.65 \\
\text{Dra} & 999 & 5.24 & 24628.56 & \text{Resolved} & 9.60_{0.25}^{0.27} \\
\text{Eri4} & 19 & 4.30 & 69.62 & \text{Resolved} & 4.11_{0.69}^{1.20} \\
\text{Herc} & 43 & 4.65 & 13.07 & \text{Resolved} & 2.25_{0.70}^{0.70} \\
\text{Hyd2} & 12 & 4.05 & 0.00 & \text{Unresolved} & <4.02 \\
\text{Leo1} & 795 & 5.22 & 19749.61 & \text{Resolved} & 9.15_{0.23}^{0.30} \\
\text{Leo2} & 305 & 5.11 & 4801.95 & \text{Resolved} & 7.35_{0.32}^{0.30} \\
\text{Leo4} & 24 & 4.42 & 37.59 & \text{Resolved} & 3.48_{0.91}^{0.59} \\
\text{Leo5} & 16 & 4.21 & 21.47 & \text{Resolved} & 2.86_{0.83}^{1.25} \\
\text{Leo6} & 9 & 3.86 & 6.81 & \text{Resolved} & 3.13_{0.95}^{2.02} \\
\text{Peg3} & 15 & 4.18 & 8.09 & \text{Resolved} & 2.99_{1.18}^{0.86} \\
\text{Peg4} & 24 & 4.42 & 41.32 & \text{Resolved} & 3.02_{0.91}^{1.12} \\
\text{Pisc2} & 12 & 4.05 & 8.16 & \text{Resolved} & 3.49_{1.01}^{2.02} \\
\text{Seg1} & 60 & 4.76 & 31.39 & \text{Resolved} & 3.65_{0.56}^{1.14} \\
\text{Seg2} & 30 & 4.51 & 0.02 & \text{Unresolved} & <2.28 \\
\text{Tri2} & 7 & 3.68 & -0.00 & \text{Unresolved} & <4.92 \\
\text{UMa1} & 37 & 4.59 & 285.48 & \text{Resolved} & 6.73_{0.70}^{1.54} \\
\text{UMa2} & 64 & 4.78 & 151.41 & \text{Resolved} & 6.46_{1.15}^{0.92} \\
\text{UMi} & 832 & 5.23 & 16155.03 & \text{Resolved} & 8.98_{0.26}^{0.27} \\
\text{W1} & 53 & 4.72 & 264.78 & \text{Resolved} & 4.80_{0.65}^{0.83} \\
\end{array}
$$

# Other Datasets from Andrew Pace

In [94]:
galaxy_names = [
    "carina_2",
    "carina_3",
    "centaurus_1",
    "grus_1",
    "grus_2",
    "horologium_1",
    "hydrus_1",
    "phoenix_2",
    "reticulum_2",
    "reticulum_3",
    "tucana_2",
    "tucana_3",
    "tucana_4",
    "tucana_5",
    "antlia_2",
    "carina_1",
    "crater_2",
    "eridanus_2",
    "fornax_1",
    "sculptor_1",
    "sextans_1",
]

short_names = [
    "Car2",
    "Car3",
    "Cen1",
    "Gru1",
    "Gru2",
    "Hor1",
    "Hyi",
    "Phe2",
    "Ret2",
    "Ret3",
    "Tuc2",
    "Tuc3",
    "Tuc4",
    "Tuc5",
    "Ant2",
    "Car1",
    "Cra2",
    "Eri2",
    "For",
    "Scl",
    "Sext",
]

galaxy_files = {
    name: os.path.join("data_ufds", f"{name}.npy")
    for name in galaxy_names
}

per_system = {}
for i,short in enumerate(short_names):
    name=galaxy_names[i]
    dataset=np.load(os.path.join("data_ufds", f"{name}.npy"))
    vel = dataset['vlos']
    verr = dataset['vlos_error']
    per_system[short] = {'v': vel, 'verr': verr, 'N': int(len(vel))}

rows = []
for gal in short_names:
    if gal not in per_system:
        continue
    rows.append(summarize_one_system(gal, per_system[gal]['v'], per_system[gal]['verr']))

summary_2 = pd.DataFrame(rows, columns=['galaxy', 'N', 'threshold', 'lhr', 'decision', 'dispersion'])

print(f"Loaded {len(galaxy_names)} galaxy files.")

/Users/pierrethibodeaux/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/Users/pierrethibodeaux/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/Users/pierrethibodeaux/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/Users/pierrethibodeaux/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/Users/pierrethibodeaux/anaconda3/lib/python3.11/site-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
/Users/pierrethibodeaux/anaconda3/lib/py

Loaded 21 galaxy files.


In [95]:
summary_2 = summary_2.sort_values('galaxy', ascending=True).reset_index(drop=True)
#print(summary[['galaxy', 'N', 'threshold', 'lhr', 'resolved', 'dispersion_summary']].to_string(index=False))
for line in summary_2[['galaxy', 'N', 'threshold', 'lhr', 'decision', 'dispersion']].to_string(index=False).splitlines()[1:]:

    stuff = line.split()
    stuff[0]="\\text{" + stuff[0] + "}"
    stuff[4]="\\text{" + stuff[4] + "}"
    print(" & ".join(stuff),"\\\\")

\text{Ant2} & 284 & 5.10 & 2916.47 & \text{Resolved} & 6.39_{0.34}^{0.28} \\
\text{Car1} & 1264 & 5.26 & 82331.55 & \text{Resolved} & 6.65_{0.14}^{0.16} \\
\text{Car2} & 17 & 4.25 & 1000.37 & \text{Resolved} & 3.78_{0.73}^{1.00} \\
\text{Car3} & 5 & 3.39 & 46.48 & \text{Resolved} & 4.17_{1.81}^{2.17} \\
\text{Cen1} & 32 & 4.54 & 420.58 & \text{Resolved} & 4.00_{0.43}^{0.68} \\
\text{Cra2} & 141 & 4.97 & 78.47 & \text{Resolved} & 2.27_{0.24}^{0.34} \\
\text{Eri2} & 92 & 4.88 & 673.43 & \text{Resolved} & 7.69_{0.83}^{1.08} \\
\text{For} & 3465 & 5.32 & 1939485.61 & \text{Resolved} & 12.09_{0.14}^{0.16} \\
\text{Gru1} & 6 & 3.55 & 16.10 & \text{Resolved} & 2.33_{0.89}^{1.24} \\
\text{Gru2} & 21 & 4.35 & 1.85 & \text{Unresolved} & <1.85 \\
\text{Hor1} & 5 & 3.39 & 136.25 & \text{Resolved} & 5.14_{1.43}^{2.70} \\
\text{Hyi} & 30 & 4.51 & 400.22 & \text{Resolved} & 3.03_{0.39}^{0.58} \\
\text{Phe2} & 5 & 3.39 & 6.02 & \text{Resolved} & 9.67_{4.56}^{3.86} \\
\text{Ret2} & 25 & 4.43 & 97.42 & 

# Other Dataset Summary

%%latex

$$
\begin{array}{c|cccrc}
\text{UFD} & N & \text{Threshold} & \text{LHR} & \text{Decision} & \text{Dispersion} \\ \hline
\text{Ant2} & 284 & 5.10 & 2916.47 & \text{Resolved} & 6.32_{0.26}^{0.34} \\
\text{Car1} & 1264 & 5.26 & 82331.55 & \text{Resolved} & 6.65_{0.13}^{0.16} \\
\text{Car2} & 17 & 4.25 & 1000.37 & \text{Resolved} & 3.78_{0.67}^{1.14} \\
\text{Car3} & 5 & 3.39 & 46.48 & \text{Resolved} & 3.69_{1.35}^{2.61} \\
\text{Cen1} & 32 & 4.54 & 420.58 & \text{Resolved} & 4.09_{0.44}^{0.62} \\
\text{Cra2} & 141 & 4.97 & 78.47 & \text{Resolved} & 2.24_{0.29}^{0.29} \\
\text{Eri2} & 92 & 4.88 & 673.43 & \text{Resolved} & 7.79_{0.90}^{1.06} \\
\text{For} & 3465 & 5.32 & 1939485.61 & \text{Resolved} & 12.12_{0.15}^{0.16} \\
\text{Gru1} & 6 & 3.55 & 16.10 & \text{Resolved} & 2.25_{0.77}^{1.22} \\
\text{Gru2} & 21 & 4.35 & 1.85 & \text{Unresolved} & <1.87 \\
\text{Hor1} & 5 & 3.39 & 136.25 & \text{Resolved} & 5.70_{1.77}^{2.67} \\
\text{Hyi} & 30 & 4.51 & 400.22 & \text{Resolved} & 3.14_{0.49}^{0.50} \\
\text{Phe2} & 5 & 3.39 & 6.02 & \text{Resolved} & 9.02_{3.83}^{4.53} \\
\text{Ret2} & 25 & 4.43 & 97.42 & \text{Resolved} & 3.50_{0.51}^{0.79} \\
\text{Ret3} & 3 & 2.85 & -0.00 & \text{Unresolved} & <11.22 \\
\text{Scl} & 2681 & 5.31 & 1040141.28 & \text{Resolved} & 9.96_{0.16}^{0.13} \\
\text{Sext} & 745 & 5.22 & 79667.73 & \text{Resolved} & 8.27_{0.24}^{0.23} \\
\text{Tuc2} & 18 & 4.28 & 211.12 & \text{Resolved} & 3.44_{0.60}^{0.77} \\
\text{Tuc3} & 26 & 4.45 & 2.73 & \text{Unresolved} & <1.70 \\
\text{Tuc4} & 9 & 3.86 & 129.72 & \text{Resolved} & 4.40_{1.06}^{1.36} \\
\text{Tuc5} & 4 & 3.17 & 2.06 & \text{Unresolved} & <3.79 \\
\end{array}
$$

In [96]:
# Combine tables

total_summary = pd.concat([summary,summary_0, summary_2], ignore_index=True, sort=False)
total_summary = total_summary.sort_values('galaxy', ascending=True).reset_index(drop=True)

# if no value make source = "\citet{}"
total_summary['source'] = total_summary['source'].fillna("\citet{}")

for line in total_summary[['galaxy','instrument','source', 'N', 'threshold', 'lhr', 'decision', 'dispersion']].to_string(index=False).splitlines()[1:]:

    stuff = line.split()
    #stuff[0]="\\text{" + stuff[0] + "}"
    #stuff[4]="\\text{" + stuff[4] + "}"
    #print(" & ".join(stuff),"\\\\")
    print(stuff[0],"&",stuff[1],"&",stuff[2],"&",stuff[3],"&",stuff[4],"&",stuff[5],"&",stuff[6],"&",stuff[7],"\\\\")

Ant2 & NaN & \citet{} & 284 & 5.10 & 2916.47 & Resolved & 6.39_{0.34}^{0.28} \\
Aqr2 & Keck/DEIMOS & \citet{} & 8 & 3.78 & 26.69 & Resolved & 4.29_{1.20}^{2.50} \\
Aqr3 & Keck/DEIMOS & \citet{} & 11 & 4.00 & 0.00 & Unresolved & <3.22 \\
Boo1 & Keck/DEIMOS & \citet{} & 92 & 4.88 & 118.57 & Resolved & 3.30_{0.49}^{0.49} \\
Boo2 & Keck/DEIMOS & \citet{} & 17 & 4.25 & 7.65 & Resolved & 1.69_{0.49}^{0.92} \\
Boo3 & Keck/DEIMOS & \citet{} & 16 & 4.21 & 7.52 & Resolved & 4.96_{1.72}^{2.06} \\
CB & Keck/DEIMOS & \citet{} & 82 & 4.85 & 111.84 & Resolved & 3.40_{0.50}^{0.61} \\
CVn1 & Keck/DEIMOS & \citet{} & 254 & 5.08 & 3319.30 & Resolved & 7.56_{0.37}^{0.54} \\
CVn2 & Keck/DEIMOS & \citet{} & 30 & 4.51 & 180.34 & Resolved & 5.57_{1.13}^{0.95} \\
Car1 & NaN & \citet{} & 1264 & 5.26 & 82331.55 & Resolved & 6.65_{0.14}^{0.16} \\
Car2 & NaN & \citet{} & 17 & 4.25 & 1000.37 & Resolved & 3.78_{0.73}^{1.00} \\
Car3 & NaN & \citet{} & 5 & 3.39 & 46.48 & Resolved & 4.17_{1.81}^{2.17} \\
Cen1 & NaN & \